# Personal Finance Analyzer -- Exploratory Analysis

This notebook is the *lab* -- it's where we poke at the data, sanity-check
the categorization rules, and look at charts before anything gets
hardened into `finance.py`. Every function called here lives in the
`analyzer/` package, not in this notebook -- the notebook only orchestrates
and visualizes.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from analyzer.utils import load_transactions
from analyzer.categorize import add_categories
from analyzer import statistics as stats_mod

sns.set_theme(style="whitegrid")
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

## 1. Load + categorize

In [ ]:
df = load_transactions("data/bank_transactions_dummy.csv")
df = add_categories(df)
print(df.shape)
df.head()

## 2. Sanity-check the categorization rules

Before trusting any chart downstream, check how much of the data actually
got categorized, and eyeball a sample of each category. If `Uncategorized`
is a large slice, the keyword rules in `analyzer/categorize.py` need more
entries -- go add them and re-run this cell.

In [ ]:
df["category"].value_counts()

In [ ]:
uncategorized = df[df["category"] == "Uncategorized"]
print(f"{len(uncategorized)} uncategorized rows ({len(uncategorized) / len(df):.1%})")
uncategorized[["date", "narration", "amount"]].head(10)

## 3. Monthly spend by category

In [ ]:
monthly = stats_mod.monthly_category_spend(df)
monthly.tail()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
monthly.plot(kind="bar", stacked=True, ax=ax, colormap="tab20")
ax.set_title("Monthly Spend by Category")
ax.set_ylabel("Amount (Rs)")
ax.legend(loc="upper left", bbox_to_anchor=(1.0, 1.0), fontsize=8)
plt.tight_layout()
plt.show()

## 4. Where does the money actually go? Top-10 merchants

In [ ]:
top10 = stats_mod.top_merchants(df, n=10)
top10

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
sns.barplot(data=top10, y="merchant", x="total_spent", hue="merchant", legend=False, ax=ax, palette="flare")
ax.set_title("Top 10 Merchants by Spend")
ax.set_xlabel("Total spend (Rs)")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

## 5. Outlier detection

Flag any single transaction more than 2 standard deviations above its
own category's mean spend. This is *within-category* z-scoring on purpose:
a Rs 3,000 grocery run is unremarkable, but a Rs 3,000 "daily micro" spend
(chai, parking, auto fare) is very unusual -- comparing both against one
global mean would hide the second and over-flag the first.

In [ ]:
outliers = stats_mod.detect_outliers(df, z_thresh=2.0)
print(f"{len(outliers)} outlier transactions found")
outliers.head(15)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
spend_only = df[(df["amount"] < 0) & (df["category"] != "Income")].copy()
spend_only["spend"] = -spend_only["amount"]
sns.stripplot(data=spend_only, x="category", y="spend", alpha=0.3, ax=ax)
sns.stripplot(data=outliers, x="category", y="spend", color="red", size=7, ax=ax, label="outlier")
ax.set_title("Spend Distribution by Category (outliers in red)")
ax.tick_params(axis="x", rotation=75)
plt.tight_layout()
plt.show()

## 6. Headline summary + savings trend

In [ ]:
stats_mod.summary_stats(df)

In [ ]:
monthly_income = df[df["amount"] > 0].groupby("month")["amount"].sum()
monthly_spend = monthly.sum(axis=1)
savings = (monthly_income - monthly_spend).reindex(monthly_spend.index)

fig, ax = plt.subplots(figsize=(14, 5))
savings.plot(kind="line", marker="o", ax=ax, color="#2E7D32")
ax.axhline(0, color="gray", linewidth=1, linestyle="--")
ax.set_title("Net Savings by Month")
ax.set_ylabel("Rs")
plt.tight_layout()
plt.show()

## Next steps

- If `Uncategorized` was non-trivial in section 2, add keywords to
  `analyzer/categorize.py` and re-run from the top.
- Once this all looks right, `python finance.py --input data/bank_transactions_dummy.csv --report`
  produces the same analysis as a shareable PDF -- that's the CLI/product
  version of everything explored here.